# F8. Quantize, evaluate, record

The end-to-end path on a small model. Steps: predict, download, convert to 16-bit GGUF, compute an
importance matrix, quantize, then measure KL divergence and peak VRAM against the 16-bit reference.
Every predicted number is written next to the measured one in `runs/<run>/manifest.json`.

Needs: `.tools/llama.cpp` and `uv sync --extra llamacpp`. Takes a few minutes for Qwen3-0.6B.

In [ ]:
from rightsize.execution import quantize_model

m = quantize_model(
    "Qwen/Qwen3-0.6B", ["Q4_K_M", "Q8_0"], imatrix=True, evaluate=True, eval_chunks=20
)
m.status, m.gate

## Predicted vs measured

In [ ]:
for step in m.steps:
    for meas in step.measurements:
        if meas.predicted is not None:
            err = (meas.value - meas.predicted) / meas.predicted * 100
            print(
                f"{step.recipe_id:22s} {meas.kind:14s} {meas.note or '':16s} predicted {meas.predicted:8.3f}  measured {meas.value:8.3f}  {err:+.1f}%"
            )

## The quality gate

In [ ]:
import json

json.dumps(
    {q: {k: v for k, v in g.items() if k != "thresholds"} for q, g in m.gate.items()}, indent=2
)

Measurements also append to `runs/measurements.jsonl`, the local seed for the calibration loop (F9). Nothing is uploaded.

In [ ]:
from pathlib import Path

print(Path("runs/measurements.jsonl").read_text()[:800])